# Interactive Figure 2 — Formation-Channel Summary

Interactive Plotly version of Figure 2: all compiled BBH, BHNS, and BNS models shown as
scatter points on three horizontal lanes as a function of their CE / no-CE fraction.

**Hover** over any point to see the paper, model name, merger rate, and the three
simple fractions (without CE, with CE, not specified).

**Output:** two HTML files saved to `interactive_figures_and_tables/`:
- `Fig2_withoutCE_fraction_interactive.html`  (x = fraction forming without CE)
- `Fig2_withCE_fraction_interactive.html`     (x = fraction forming with CE)

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
path_github  = "/Users/floorbroekgaarden/Projects/GitHub/Rates_of_Formation_Channels/"
path_fc_data = path_github + "fc_data/Data_formation_channels_intrinsic/"
path_table   = path_github + "interactive_figures_and_tables/formation_channel_rates_table.csv"
path_save    = path_github + "interactive_figures_and_tables/"

# Column names in the rates CSVs
COL_RATE    = "All intrinsic (z=0) [Gpc^-3 yr^-1]"
COL_NO_CE   = "fraction without common envelope"
COL_WITH_CE = "fraction with common envelope"
COL_OTHER   = "fraction not specified"

In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────

def load_dco(dco_label, table_df):
    """
    Load the rates CSV for one DCO type and left-join with the summary table
    to attach the Paper name to each model row.
    """
    df = pd.read_csv(path_fc_data + dco_label + "_rates_review.csv")

    # numeric coercion for fraction columns (some may be stored as strings)
    for col in [COL_RATE, COL_NO_CE, COL_WITH_CE, COL_OTHER]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df.fillna(0.0, inplace=True)

    # join to get Paper name
    ref = table_df[["Model", "Paper"]].copy()
    ref["Model"] = ref["Model"].astype(str).str.strip()
    df["_key"] = df["model"].astype(str).str.strip()
    df = df.merge(ref, left_on="_key", right_on="Model", how="left")
    df["Paper"] = df["Paper"].fillna("N/A")
    df["DCO"] = dco_label

    return df

In [ ]:
# ── Hover text builder ────────────────────────────────────────────────────────

def hover_text(row):
    """Build the HTML tooltip string for one model row."""
    paper   = row.get("Paper",  "N/A")
    model   = row.get("model",  "N/A")
    rate    = row.get(COL_RATE,    float("nan"))
    no_ce   = row.get(COL_NO_CE,   float("nan"))
    with_ce = row.get(COL_WITH_CE, float("nan"))
    other   = row.get(COL_OTHER,   float("nan"))

    def f4(v):
        return f"{v:.4f}" if pd.notna(v) and v != 0.0 or v == 0.0 else "N/A"

    rate_s = f"{rate:.2f}" if pd.notna(rate) else "N/A"

    return (
        f"<b>Paper:</b> {paper}<br>"
        f"<b>Model:</b> {model}<br>"
        f"<b>──────────────────────────</b><br>"
        f"<b>Rate [Gpc\u207b\u00b3 yr\u207b\u00b9]:</b> {rate_s}<br>"
        f"fraction without CE: {f4(no_ce)}<br>"
        f"fraction with CE: {f4(with_ce)}<br>"
        f"fraction not specified: {f4(other)}"
    )

In [ ]:
# ── Main interactive figure function ─────────────────────────────────────────

# Vertical lane positions and styling per DCO type
DCO_CFG = {
    "BH-BH": {"y": 2.0, "color": "#1a1a1a", "label": "BBH"},
    "BH-NS": {"y": 1.0, "color": "#1f77b4", "label": "BHNS"},
    "NS-NS": {"y": 0.0, "color": "#e8a800", "label": "BNS"},
}


def make_fig2_interactive(
    x_col=COL_NO_CE,
    jitter_scale=0.05,
    point_size=9,
    alpha=0.82,
    save_path=None,
    show=True,
):
    """
    Build an interactive version of Figure 2.

    Parameters
    ----------
    x_col : str
        Which fraction column to use as the x-axis
        (COL_NO_CE  = 'fraction without common envelope', or
         COL_WITH_CE = 'fraction with common envelope').
    save_path : str or None
        If provided, save the figure as an HTML file at this path.
    show : bool
        If True, display the figure inline.
    """
    table_df = pd.read_csv(path_table)
    rng = np.random.default_rng(42)   # fixed seed → reproducible jitter

    fig = go.Figure()

    # ── Horizontal guide lines and shaded lanes ───────────────────────────────
    for cfg in DCO_CFG.values():
        yc = cfg["y"]
        # shaded band
        fig.add_shape(
            type="rect", x0=0, x1=1, y0=yc - 0.25, y1=yc + 0.25,
            fillcolor="rgba(180,180,180,0.07)", line=dict(width=0), layer="below",
        )
        # centre line
        fig.add_shape(
            type="line", x0=0, x1=1, y0=yc, y1=yc,
            line=dict(color="rgba(120,120,120,0.35)", width=1.2), layer="below",
        )

    # ── One scatter trace per DCO type ───────────────────────────────────────
    for dco_label, cfg in DCO_CFG.items():
        df = load_dco(dco_label, table_df)

        # keep only rows with a valid, physical x value
        df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
        df = df.dropna(subset=[x_col])
        df = df[(df[x_col] >= 0) & (df[x_col] <= 1)].reset_index(drop=True)

        x_vals = df[x_col].to_numpy()
        y_vals = np.full(len(df), cfg["y"]) + rng.uniform(
            -jitter_scale, jitter_scale, size=len(df)
        )

        tooltips = [hover_text(row) for _, row in df.iterrows()]

        fig.add_trace(
            go.Scatter(
                x=x_vals.tolist(),
                y=y_vals.tolist(),
                mode="markers",
                name=cfg["label"],
                marker=dict(
                    color=cfg["color"],
                    size=point_size,
                    opacity=alpha,
                    line=dict(width=0.6, color="black"),
                ),
                customdata=tooltips,
                hovertemplate="%{customdata}<extra></extra>",
            )
        )

        # model count annotation to the right of each lane
        fig.add_annotation(
            x=1.03, y=cfg["y"],
            xref="x", yref="y",
            text=f"<b>{len(df)}</b> models",
            showarrow=False,
            xanchor="left", yanchor="middle",
            font=dict(size=11, color="#555"),
        )

    # ── Invisible dummy trace to activate the secondary (top) x-axis ─────────
    # xaxis2 has range [1, 0], so it shows the complementary fraction.
    fig.add_trace(
        go.Scatter(
            x=[0, 1], y=[None, None],
            xaxis="x2", yaxis="y",
            mode="markers",
            marker=dict(opacity=0, size=0),
            showlegend=False,
            hoverinfo="skip",
            name="",
        )
    )

    # ── Axis labels ───────────────────────────────────────────────────────────
    if x_col == COL_NO_CE:
        x_title  = "Fraction forming <b>without</b> CE"
        x2_title = "Fraction forming <b>with</b> CE"
    else:
        x_title  = "Fraction forming <b>with</b> CE"
        x2_title = "Fraction forming <b>without</b> CE"

    tick_vals = [round(i / 10, 1) for i in range(11)]

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        height=400,
        width=1100,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=70, r=160, t=70, b=60),
        hoverlabel=dict(bgcolor="white", font_size=11, namelength=-1),
        legend=dict(
            x=1.01, y=0.5,
            xanchor="left", yanchor="middle",
            font=dict(size=13),
            borderwidth=1, bordercolor="#e0e4ef",
        ),
        # ── primary (bottom) x-axis ──
        xaxis=dict(
            range=[0, 1],
            title=dict(text=x_title, font=dict(size=14)),
            tickvals=tick_vals,
            tickformat=".1f",
            tickfont=dict(size=12),
            showgrid=True,
            gridcolor="rgba(0,0,0,0.10)",
            gridwidth=1,
            minor=dict(
                tickvals=[round(i / 20, 2) for i in range(21)],
                showgrid=True,
                gridcolor="rgba(0,0,0,0.04)",
            ),
            zeroline=False,
            showline=True,
            linecolor="#aaa",
            linewidth=1.5,
        ),
        # ── secondary (top) x-axis shows complementary fraction ──
        xaxis2=dict(
            overlaying="x",
            side="top",
            range=[1, 0],          # reversed so 0→1 on top matches 1→0 of main
            title=dict(text=x2_title, font=dict(size=12, color="#777")),
            tickvals=tick_vals,
            tickformat=".1f",
            tickfont=dict(size=11, color="#888"),
            showgrid=False,
            zeroline=False,
            showline=True,
            linecolor="#ccc",
            linewidth=1,
        ),
        # ── y-axis: three categorical lanes ──
        yaxis=dict(
            range=[-0.45, 2.45],
            tickvals=[0, 1, 2],
            ticktext=["BNS", "BHNS", "BBH"],
            tickfont=dict(size=14, color="#222"),
            showgrid=False,
            zeroline=False,
            showline=True,
            linecolor="#aaa",
            linewidth=1.5,
        ),
    )

    if save_path:
        fig.write_html(save_path)
        print(f"Saved: {save_path}")

    if show:
        fig.show()

    return fig

## Figure 2a — x-axis: fraction forming **without** CE

In [ ]:
fig_no_ce = make_fig2_interactive(
    x_col=COL_NO_CE,
    save_path=path_save + "Fig2_withoutCE_fraction_interactive.html",
)

## Figure 2b — x-axis: fraction forming **with** CE

In [ ]:
fig_with_ce = make_fig2_interactive(
    x_col=COL_WITH_CE,
    save_path=path_save + "Fig2_withCE_fraction_interactive.html",
)